# Task 2 — Model Building & Training: creditcard
**Adey Innovations Inc. — Fraud Detection Project**

This notebook covers:
1. Load processed data from Task 1
2. Stratified train-test split + SMOTE (training only)
3. Baseline model — Logistic Regression
4. Ensemble model — Random Forest
5. Stratified 5-Fold cross-validation
6. Model comparison & selection
7. Save models for Task 3

> **Note:** This dataset has extreme imbalance (0.17% fraud), so AUC-PR is the primary metric — it's far more informative than accuracy or ROC-AUC under severe imbalance.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import split_and_resample
from modeling import (
    train_logistic_regression, train_random_forest,
    evaluate_model, plot_confusion_matrix, plot_precision_recall_curve,
    cross_validate_model, build_comparison_table, save_model
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Processed Data

In [2]:
cc = pd.read_csv('../data/processed/creditcard_processed.csv')
print('Shape:', cc.shape)
cc.head()

Shape: (283726, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-1.996823,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.244200,0
1,-1.996823,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.342584,0
2,-1.996802,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,1.158900,0
3,-1.996802,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.139886,0
4,-1.996781,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,-0.073813,0


## 2. Stratified Train-Test Split + SMOTE

In [3]:
X_train, X_test, y_train, y_test = split_and_resample(cc, target_col='Class')

print('X_train:', X_train.shape, '  y_train balance:', y_train.value_counts().to_dict())
print('X_test :', X_test.shape, '  y_test balance :', y_test.value_counts().to_dict())

INFO: Train size: 226980, Test size: 56746
INFO: Class distribution before SMOTE:
Class
0    226602
1       378
Name: count, dtype: int64
INFO: Class distribution after SMOTE:
Class
0    226602
1    226602
Name: count, dtype: int64


X_train: (453204, 30)   y_train balance: {0: 226602, 1: 226602}
X_test : (56746, 30)   y_test balance : {0: 56651, 1: 95}


## 3. Baseline Model — Logistic Regression

In [4]:
log_reg = train_logistic_regression(X_train, y_train)

lr_results = evaluate_model(log_reg, X_test, y_test, model_name='Logistic Regression')
lr_results['y_test'] = y_test

plot_confusion_matrix(lr_results['confusion_matrix'], model_name='Logistic Regression', dataset_name='Credit Card')

INFO: Logistic Regression trained.
INFO: Logistic Regression -> AUC-PR: 0.6768, F1: 0.1002
INFO: Confusion Matrix:
[[55172  1479]
 [   12    83]]


  Saved → outputs/confusion_matrix_logistic_regression_credit_card.png


## 4. Ensemble Model — Random Forest

In [5]:
rf = train_random_forest(X_train, y_train, n_estimators=100, max_depth=10)

rf_results = evaluate_model(rf, X_test, y_test, model_name='Random Forest')
rf_results['y_test'] = y_test

plot_confusion_matrix(rf_results['confusion_matrix'], model_name='Random Forest', dataset_name='Credit Card')

INFO: Random Forest trained (n_estimators=100, max_depth=10).
INFO: Random Forest -> AUC-PR: 0.7797, F1: 0.6553
INFO: Confusion Matrix:
[[56588    63]
 [   18    77]]


  Saved → outputs/confusion_matrix_random_forest_credit_card.png


In [6]:
plot_precision_recall_curve([lr_results, rf_results], dataset_name='Credit Card')

  Saved → outputs/precision_recall_credit_card.png


## 5. Stratified 5-Fold Cross-Validation

In [7]:
print('Cross-validating Logistic Regression...')
cv_lr = cross_validate_model(
    lambda: LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    X_train, y_train, k=5
)

print('\nCross-validating Random Forest...')
cv_rf = cross_validate_model(
    lambda: RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    X_train, y_train, k=5
)

Cross-validating Logistic Regression...


INFO: Fold 1: AUC-PR=0.9918, F1=0.9456
INFO: Fold 2: AUC-PR=0.9918, F1=0.9449
INFO: Fold 3: AUC-PR=0.9922, F1=0.9470
INFO: Fold 4: AUC-PR=0.9918, F1=0.9462
INFO: Fold 5: AUC-PR=0.9922, F1=0.9482



Cross-validating Random Forest...


INFO: Fold 1: AUC-PR=0.9999, F1=0.9912
INFO: Fold 2: AUC-PR=0.9998, F1=0.9897
INFO: Fold 3: AUC-PR=0.9998, F1=0.9898
INFO: Fold 4: AUC-PR=0.9997, F1=0.9900
INFO: Fold 5: AUC-PR=0.9998, F1=0.9904


In [8]:
lr_results.update({
    'cv_auc_pr_mean': cv_lr['auc_pr_mean'], 'cv_auc_pr_std': cv_lr['auc_pr_std'],
    'cv_f1_mean': cv_lr['f1_mean'], 'cv_f1_std': cv_lr['f1_std']
})
rf_results.update({
    'cv_auc_pr_mean': cv_rf['auc_pr_mean'], 'cv_auc_pr_std': cv_rf['auc_pr_std'],
    'cv_f1_mean': cv_rf['f1_mean'], 'cv_f1_std': cv_rf['f1_std']
})

## 6. Model Comparison & Selection

In [9]:
comparison = build_comparison_table([lr_results, rf_results])
comparison

,Model,Test AUC-PR,Test F1-Score,CV AUC-PR (mean ± std),CV F1 (mean ± std)
0,Logistic Regression,0.6768,0.1002,0.9920 ± 0.0002,0.9464 ± 0.0011
1,Random Forest,0.7797,0.6553,0.9998 ± 0.0000,0.9902 ± 0.0006


> **Model Selection Justification:**
>
> For creditcard.csv, the Random Forest model is expected to substantially outperform Logistic Regression on AUC-PR, since the PCA-transformed V1-V28 features have complex, non-linear relationships with the target that tree-based splits can capture more effectively than a linear decision boundary.
>
> Given the extreme imbalance (0.17% fraud), AUC-PR is the deciding metric — it focuses on performance on the positive (fraud) class regardless of the large number of true negatives. The model with the higher AUC-PR is selected as the primary model for Task 3 SHAP analysis.
>
> *(Update this paragraph with the actual numeric comparison once you've run the cells above.)*

## 7. Save Models

In [10]:
import os
import joblib
os.makedirs('../data/models', exist_ok=True)

joblib.dump(log_reg, '../data/models/logistic_regression_creditcard.joblib')
joblib.dump(rf, '../data/models/random_forest_creditcard.joblib')

# Also save the test set for Task 3 SHAP analysis
X_test.to_csv('../data/processed/creditcard_X_test.csv', index=False)
y_test.to_csv('../data/processed/creditcard_y_test.csv', index=False)

print('✅ Models and test set saved.')

✅ Models and test set saved.


## Summary

| Model | Test AUC-PR | Test F1 | CV AUC-PR | CV F1 |
|-------|-------------|---------|-----------|-------|
| Logistic Regression | see table above | | | |
| Random Forest | see table above | | | |

**Selected model for Task 3:** Random Forest (pending confirmation from comparison table)